# Decomposition of FID into Sinusoidal Signals

The decomposition of an experimental FID into a superposition of sinusoidal signals
would deliver very useful information on frequencies, phases, amplitudes, and decay rates.
Such a list of parameters could then be compared with a database of expected signals, and
be used directly for quantification. This idea has been explored from the standpoint of
Bayesian probability theory by Bretthorst (https://www.sciencedirect.com/science/article/pii/002223649090287J}[J. Magn. Reson. 1990 ]), and applied by Krishnamurthy (https://onlinelibrary.wiley.com/doi/abs/10.1002/mrc.4022) to quantification, in combination with division of the FID into sub-FIDs corresponding 
to regions of interest in the spectrum.

Here, we explore the idea again and implement its basics in Julia.

In [ ]:
import Pkg
Pkg.activate(".")

using Revise
using Plots
using NMRflux
using JLD2
using LinearAlgebra
using LsqFit
using FFTW

import DSP

theme(:solarized_light)
default(size=(600,400),
        tick_direction=:out,
        framestyle=:box,
        margin=(6,:mm),
        legendfontsize=8,
        guidefontsize=10,
        tickfontsize=10,
        fontfamily="Helvetica",
        fmt=:svg, dpi=100)

In [ ]:
@load "processedData.jld2"

## Digital Filtering of the FID

In many cases we are only interested in a limited frequency range of the spectrum.
This can be achieved by digital filtering before frequency estimation and fitting.

An ideal band-pass filter has a rectangular frequency response, which corresponds
in the time domain to a sinc function $h(t) = \sin(\pi B t)/(\pi t)$, where $B$
is the bandwidth. Because the FID is sampled and finite, this ideal filter must be
truncated to a finite impulse response (FIR) of length $2n+1$ samples.
Abrupt truncation introduces Gibbs ringing — large oscillatory artefacts near the
band edges — because the sharp cutoff in time creates spectral leakage in frequency.
Multiplying the truncated sinc by a **window function** tapers the coefficients
smoothly to zero at both ends, suppressing the sidelobes at the cost of a slightly
wider transition band.
The **Blackman window** $w(k) = 0.42 - 0.5\cos(\pi k/n) + 0.08\cos(2\pi k/n)$
is a particularly good choice here because it achieves very low sidelobe levels
(around -58 dB), which is important in NMR where weak peaks must be resolved
next to strong solvent or lipid signals.
Its three-term cosine form exactly cancels the first two sidelobe families of the
truncated sinc, trading a modestly wider passband for a clean stopband.
The band-reject filter is constructed as the complement of the band-pass
($\delta[t] - h_{\rm bp}(t)$), passing all frequencies except the rejected band.

In [ ]:
function BandRejectFilter(lf,hf,n)
   b = [ t == 0 ? -ComplexF64(hf-lf,0.0) : 1.0/(2pi*im*t)*(exp(2pi*im*lf*t)-exp(2pi*im*hf*t)) for t=-n:n ]
   b .*= 0.42 .- 0.5*cos.(pi/n*(0:2n)) .+ 0.08*cos.(2pi/n*(0:2n))
   b[n+1] += ComplexF64(1.0,0.0)
   return b
end

function BandPassFilter(lf,hf,n)
   b = -[ t == 0 ? -ComplexF64(hf-lf,0.0) : 1.0/(2pi*im*t)*(exp(2pi*im*lf*t)-exp(2pi*im*hf*t)) for t=-n:n ]
   b .*= 0.42 .- 0.5*cos.(pi/n*(0:2n)) .+ 0.08*cos.(2pi/n*(0:2n))
   return b
end

function myfilt(f, data::SpectData{T,N}) where {T,N}
    filtered_data = DSP.filt(f, data.dat)
    return SpectData(filtered_data, NMRflux.coords(data))
end

## Frequency Estimation via ESPRIT

ESPRIT (Estimation of Signal Parameters via Rotational Invariance Techniques) is a
subspace method for estimating the frequencies of damped sinusoids in noise. It exploits
the algebraic structure of the signal subspace rather than searching over a frequency grid,
giving superresolution estimates that are not limited by the FFT bin spacing.

**Algorithm outline:**

1. **Hankel matrix.** From the FID $x[1],\ldots,x[N]$, form the $M \times L$ Hankel matrix
   ($L = \lfloor N/2 \rfloor$, $M = N - L + 1$):
   $$X_{ij} = x[i+j-1]$$
   For $K$ ideal damped sinusoids, $X$ has rank $K$ in the absence of noise.

2. **Signal subspace.** Compute the SVD of $X$. The $K$ dominant left singular vectors
   span the signal subspace $U_s$ ($M \times K$).

3. **Rotational invariance.** Split $U_s$ into two overlapping sub-matrices differing by
   one row:
   $$U_s^\uparrow = U_s[1{:}M-1,\,:\,], \qquad U_s^\downarrow = U_s[2{:}M,\,:\,]$$
   Because consecutive rows of $X$ are related by a time shift, these satisfy
   $U_s^\downarrow \approx U_s^\uparrow \Phi$, where $\Phi$ is a $K\times K$ matrix
   whose eigenvalues are $e^{(-\delta_k + i\omega_k)\Delta t}$.

4. **Frequency extraction.** Solve for $\Phi$ by least squares
   ($\Phi = (U_s^\uparrow)^+ U_s^\downarrow$) and take the angles of its eigenvalues
   to recover the frequencies $\omega_k$ in rad/sample.

**Reference:**
R. Roy and T. Kailath, "ESPRIT — Estimation of Signal Parameters via Rotational Invariance
Techniques," *IEEE Trans. Acoust., Speech, Signal Process.*, vol. 37, no. 7,
pp. 984–995, July 1989.

In [ ]:
function esprit(x, K)
    N = length(x)
    L = N ÷ 2
    M = N - L + 1

    X    = [x[i+j-1] for i in 1:M, j in 1:L]
    svdX = svd(X)
    Us   = svdX.U[:, 1:K]

    Phi = Us[1:M-1, :] \ Us[2:M, :]
    λ   = eigvals(Phi)

    ωs     = angle.(λ)       # rad/sample
    decays = -log.(abs.(λ))  # 1/sample, positive for damped sinusoids

    idx = sortperm(ωs)
    return ωs[idx], decays[idx], svdX.S
end

function esprit(data::SpectData{T,1}, K) where T
    freqs, decays, S = esprit(data.dat, K)
    dt       = step(NMRflux.coords(data, 1))
    freq_hz  = freqs ./ dt ./ (2π)
    decay_hz = decays ./ dt
    return freq_hz, decay_hz, S
end

function unique_frequency_indices(freqs, tol)
    idx    = sortperm(freqs)
    sorted = freqs[idx]
    keep   = Int[]
    group_start = 1

    for i in 2:length(sorted)
        if sorted[i] - sorted[group_start] > tol
            group_centre = (sorted[group_start] + sorted[i-1]) / 2
            best = argmin(abs(sorted[j] - group_centre) for j in group_start:i-1)
            push!(keep, idx[group_start + best - 1])
            group_start = i
        end
    end
    group_centre = (sorted[group_start] + sorted[end]) / 2
    best = argmin(abs(sorted[j] - group_centre) for j in group_start:length(sorted))
    push!(keep, idx[group_start + best - 1])

    return keep
end


## Model Selection via Laplace Approximation to the Log Evidence

To compare models with different numbers of oscillators $K$, we need the
**model evidence** $p(\mathbf{x} \mid K)$ — the probability of the data after
marginalising over all parameter values:
$$p(\mathbf{x} \mid K) = \int p(\mathbf{x} \mid \mathbf{B}, K)\, p(\mathbf{B} \mid K)\, d\mathbf{B}$$
A model with more parameters fits better, but spreads its prior over a larger space,
incurring an automatic Occam penalty. The evidence balances fit quality against
model complexity without manual regularisation.

**Laplace approximation.** For a Gaussian likelihood with noise variance $\sigma^2$
and a broad, flat prior, expanding the log-posterior to second order around the
maximum-likelihood estimate $\hat{\mathbf{B}}$ gives:
$$\ln p(\mathbf{x} \mid K) \approx
  -\frac{n}{2}\ln\sigma^2
  +\frac{1}{2}\ln\det(J^\top J)
  -\frac{K}{2}\ln n$$
where $J$ is the Jacobian of the residuals evaluated at $\hat{\mathbf{B}}$,
$n$ is the number of data points, and $K$ is the number of fitted parameters.
The three terms are respectively the log-likelihood at the optimum, the log-volume
of the posterior peak (how well the data constrain the parameters), and the BIC
penalty for model complexity.

To select the number of oscillators, fit for $K = 1, 2, \ldots$ and choose the
$K$ with the highest log evidence.

**Numerical note.** When some parameters are poorly constrained (e.g. duplicate
frequencies), $J^\top J$ becomes singular. A small ridge $\varepsilon I$ is added
before taking the log-determinant to regularise this case.

In [ ]:
const DECAY_MAX = 40π   # ~20 Hz maximum linewidth

function laplace_log_evidence(fit)
    K   = length(fit.param)
    n   = length(fit.resid)
    σ²  = sum(fit.resid.^2) / n
    JtJ = fit.jacobian' * fit.jacobian
    ε   = 1e-10 * maximum(diag(JtJ))
    return -n/2 * log(σ²) + 1/2 * logabsdet(JtJ + ε*I)[1] - K/2 * log(n)
end

## Signal Model

The FID is modelled as a superposition of $K$ damped complex sinusoids:
$$x(t) = \sum_{k=1}^{K} c_k \, e^{(-\delta_k + i\omega_k)t}$$
where $c_k = a_k e^{i\phi_k}$ is a complex amplitude encoding both the peak
amplitude $a_k$ and phase $\phi_k$, $\omega_k = 2\pi f_k$ is the angular
frequency, and $\delta_k > 0$ is the decay rate (inversely proportional to the
transverse relaxation time $T_2$).

**Why fix the frequencies?**
In principle one could fit all $4K$ parameters $(\text{Re}\,c_k,\,\text{Im}\,c_k,\,\omega_k,\,\delta_k)$
simultaneously. In practice this leads to a severely ill-conditioned optimisation
for two reasons.
First, the cost function has many nearly equivalent local minima: two oscillators
at slightly different frequencies can produce very similar residuals, so the
optimiser can swap, merge, or drift them without penalty.
Second, the Jacobian columns for $\omega_k$ and $\delta_k$ are proportional to
$c_k \cdot t \cdot e^{(-\delta_k+i\omega_k)t}$; if two oscillators converge to
the same frequency their columns become linearly dependent and the Jacobian
becomes singular.

ESPRIT provides accurate, superresolution frequency estimates directly from the
data, so there is no reason to re-estimate them by nonlinear optimisation.
Fixing $\omega_k$ at the ESPRIT values reduces the problem to $3K$ parameters
$( \text{Re}\,c_k,\,\text{Im}\,c_k,\,\delta_k )$.
For fixed frequencies the model is **linear** in the complex amplitudes $c_k$,
and only mildly nonlinear through the decay rates $\delta_k$, giving a
much better conditioned fit.

The decay rate is reparametrised as $\delta_k = \delta_{\max}\,\sigma(\gamma_k)$
where $\sigma$ is the sigmoid function, bounding $\delta_k$ strictly within
$(0, \delta_{\max})$ and preventing runaway decay during optimisation.

In [ ]:
function oscillators_fixed_freq(r, B, freq_hz)
    K        = length(freq_hz)
    freq_rads = freq_hz .* 2π
    return sum(1:K) do k
        c     = B[3k-2] + im*B[3k-1]
        decay = DECAY_MAX / (1 + exp(-B[3k]))
        c .* exp.((-decay + im*freq_rads[k]) .* r)
    end
end

function oscillators_fixed_freq_jacobian(r, B, freq_hz)
    K         = length(freq_hz)
    freq_rads = freq_hz .* 2π
    nr        = length(r)
    J         = zeros(2nr, 3K)
    for k in 1:K
        c     = B[3k-2] + im*B[3k-1]
        decay = DECAY_MAX / (1 + exp(-B[3k]))
        e     = exp.((-decay + im*freq_rads[k]) .* r)
        ce    = c .* e

        J[1:nr,     3k-2] =  real(e);  J[nr+1:end, 3k-2] = imag(e)
        J[1:nr,     3k-1] = -imag(e);  J[nr+1:end, 3k-1] = real(e)

        d_factor = decay * (1 - decay / DECAY_MAX)
        d = -r .* ce .* d_factor
        J[1:nr,     3k]   =  real(d);  J[nr+1:end, 3k]   = imag(d)
    end
    return J
end


In [ ]:
function fit_fid_fixed_freq(fid::AbstractVector{<:Complex}, r::StepRangeLen,
                             freq_hz::Vector, decay_rate::Vector)
    K  = length(freq_hz)
    dt = step(r)

    model(r, B)    = let pred = oscillators_fixed_freq(r, B, freq_hz)
        [real(pred .- fid); imag(pred .- fid)]
    end
    jacobian(r, B) = oscillators_fixed_freq_jacobian(r, B, freq_hz)

    # Initialise γ from ESPRIT decay estimates via sigmoid inverse (logit)
    decay_clamped = clamp.(decay_rate, 1e-3, DECAY_MAX - 1e-3)
    γ0 = log.(decay_clamped ./ (DECAY_MAX .- decay_clamped))

    B0          = zeros(3K)
    B0[1:3:end] .= 1.0   # initial cr
    B0[3:3:end]  = γ0

    fit = curve_fit(model, jacobian, r, zeros(2length(r)), B0)

    B          = fit.param
    amplitude  = abs.(B[1:3:end] .+ im .* B[2:3:end])
    phase_rad  = angle.(B[1:3:end] .+ im .* B[2:3:end])
    decay_rate_fit = DECAY_MAX ./ (1 .+ exp.(-B[3:3:end]))

    idx = sortperm(freq_hz)
    result = Dict(
        "n_oscillators" => K,
        "residual"      => sum(fit.resid.^2),
        "log_evidence"  => laplace_log_evidence(fit),
        "resonances"    => [
            Dict(
                "frequency_hz"  => freq_hz[i],
                "amplitude"     => amplitude[i],
                "phase_rad"     => phase_rad[i],
                "decay_rate_hz" => decay_rate_fit[i] / (2π),
                "T2_s"          => π / decay_rate_fit[i]
            )
            for i in idx
        ]
    )

    return fit, result
end

function fit_fid_fixed_freq(fid::SpectData{T,1}, freq_hz::Vector, decay_rate::Vector) where T
    fit, result = fit_fid_fixed_freq(fid.dat, NMRflux.coords(fid, 1), freq_hz, decay_rate)
    return fit, result
end


In [ ]:
dt = step(NMRflux.coords(fidData,1))
SW = 1/dt
H2OReject = BandRejectFilter(-60.0/SW,250.0/SW,1000)
aromaticSelect = BandPassFilter(500.0/SW,2400/SW,1000)

fidDataF = myfilt(H2OReject,fidData)

In [ ]:
δt = step(NMRflux.coords(fidDataF,1))
fid = fidDataF[1200:7200,1]
fid_data = fid.dat
r = NMRflux.coords(fid,1)

freq_hz, decays, _  = esprit(fid , 150)
fit, result = fit_fid_fixed_freq(fid,freq_hz,decays)

In [ ]:
import JSON
open("fit_result.json", "w") do io
    JSON.print(io, result)
end


In [ ]:
model_pred = oscillators_fixed_freq(r, fit.param, freq_hz)
plot(r, real(fid_data), label="Data",opacity=0.5,xlabel="Time (s)", ylabel="Signal (a.u.)") 
plot!(r, real(model_pred), label="Fit",opacity=0.5)
plot!(r, real(fid_data) - real(model_pred), label="Residual",opacity=0.9)

In [ ]:
Plots.plot(pi ./ decays, pi .* (1 .+ exp.(-fit.param[3:3:end]))./DECAY_MAX, seriestype=:scatter, xlabel="T2 (1/s)", ylabel="Fitted T2 (1/s)", title="Fitted vs Estimated T2",aspect_ratio=1,
xlims=(0,5),ylims=(0,5))

In [ ]:
modelsizes=[1, 2, 3, 4, 5, 10, 20, 50, 100, 150, 200, 300, 500]
evidences = Float64[]
# modelsizes=[300,500]

for K in modelsizes
    freq_hz, _ = esprit(fid , K)
    fit, result = fit_fid_fixed_freq(fid,freq_hz)
    push!(evidences, result["log_evidence"])
end
plot(modelsizes, evidences, marker=:o, label="Log Evidence", xlabel="Number of Oscillators", ylabel="Log Evidence")

In [ ]:
import NMRflux.coords
coords(fidData,1)  # check that the file names are stored as 2nd dimension coordinates

In [ ]:
1/17